# Plant Disease Classification - Colab workflow

This notebook downloads PlantVillage from TensorFlow Datasets during the Colab session; you do not need to upload dataset files from your computer. It creates a reproducible 70/15/15 train/validation/test split, explores image properties, trains an augmented MobileNetV2 transfer-learning model, compares frozen-feature and fine-tuned results, evaluates both stages on the held-out test set, and predicts one test image.

## Run in Google Colab

1. Push this repository to GitHub and open this notebook in Colab (File > Open notebook > GitHub), or upload only the notebook file.
2. In Colab, choose **Runtime > Change runtime type > T4 GPU** if available. Training on CPU is much slower; the notebook will report the available device.
3. Run the cells from top to bottom. The setup cell clones the repository and installs missing packages. The acquisition cell downloads PlantVillage through TFDS and exports at most 200 images per class into the temporary Colab VM.
4. Allow several minutes for dataset download and model training. The 200-image-per-class cap matches the documented sample design, but TFDS downloads its source dataset before the subset is exported.

Colab storage is temporary. Download the model/metrics or copy them to Google Drive before the runtime is deleted if you need to keep them. The test set is for final reporting only; model selection uses validation loss.

In [ ]:
import importlib.util
from pathlib import Path
import subprocess
import sys

try:
    import google.colab  # noqa: F401
except ImportError as error:
    raise RuntimeError(
        'Run this notebook in Google Colab so it can clone the project and '
        'download the dataset into the session.'
    ) from error

REPOSITORY_URL = 'https://github.com/AHIRWE1/plant-disease-classification.git'
PROJECT_DIRECTORY = Path('/content/plant-disease-classification')
if not (PROJECT_DIRECTORY / 'src').is_dir():
    subprocess.run(
        ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_DIRECTORY)],
        check=True
    )
sys.path.insert(0, str(PROJECT_DIRECTORY))

packages = {
    'tensorflow': 'tensorflow',
    'tensorflow_datasets': 'tensorflow-datasets',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'PIL': 'pillow'
}
missing_packages = [
    package for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', *missing_packages],
        check=True
    )

import tensorflow as tf
print('Project directory:', PROJECT_DIRECTORY)
print('TensorFlow:', tf.__version__)
print('Visible GPU devices:', tf.config.list_physical_devices('GPU'))

## 1. Download and inspect PlantVillage

The acquisition utility uses TensorFlow Datasets' `plant_village` dataset. It downloads/caches the dataset within this Colab runtime and exports a seeded sample of up to 200 images for each of the 38 disease/healthy classes. No local dataset upload is involved. Keep the cap and seed fixed when reproducing this experiment.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

from src.data_acquisition import acquire_plant_village_dataset
from src.preprocessing import create_split_manifest, load_split_datasets
from src.visualization import generate_dataset_visualizations
from src.model import build_model, compile_model, unfreeze_last_layers
from src.training import evaluate_model
from src.prediction import load_predictor

RANDOM_SEED = 42
MAX_IMAGES_PER_CLASS = 200
BATCH_SIZE = 16
DATASET_METADATA = PROJECT_DIRECTORY / 'data' / 'raw' / 'dataset_metadata.json'

if not DATASET_METADATA.is_file():
    acquisition_report = acquire_plant_village_dataset(
        PROJECT_DIRECTORY,
        max_images_per_class=MAX_IMAGES_PER_CLASS,
        random_seed=RANDOM_SEED
    )
    print(acquisition_report)
else:
    print('Using dataset already downloaded in this Colab session.')

with DATASET_METADATA.open(encoding='utf-8') as file:
    dataset_metadata = json.load(file)
print(
    f"Loaded {dataset_metadata['total_images']:,} images across "
    f"{dataset_metadata['number_of_classes']} classes."
)

In [ ]:
class_counts = pd.Series(dataset_metadata['images_per_class']).sort_values()
display(class_counts.to_frame('image_count'))

axis = class_counts.plot(kind='barh', figsize=(10, 12), color='forestgreen')
axis.set_title('PlantVillage subset: images per class')
axis.set_xlabel('Images')
plt.tight_layout()
plt.show()

## 2. Preprocess and create reproducible splits

Images are decoded as RGB, resized to 224 x 224, and preprocessed using MobileNetV2's `preprocess_input`. A fixed-seed, stratified 70/15/15 manifest assigns paths to train, validation, and test without copying the images. Augmentation will be applied to training batches only; validation and test images remain unchanged.

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)
split_report = create_split_manifest(
    PROJECT_DIRECTORY,
    train_fraction=0.70,
    validation_fraction=0.15,
    test_fraction=0.15,
    random_seed=RANDOM_SEED
)
print('Images per split:', split_report['images_per_split'])

datasets = load_split_datasets(
    PROJECT_DIRECTORY,
    image_size=(224, 224),
    batch_size=BATCH_SIZE,
    random_seed=RANDOM_SEED
)
images, labels = next(iter(datasets['train']))
print('Training batch:', images.shape, labels.shape)
print(
    'Preprocessed pixel range:',
    float(tf.reduce_min(images)),
    float(tf.reduce_max(images))
)

In [ ]:
visualization_report = generate_dataset_visualizations(
    PROJECT_DIRECTORY,
    random_seed=RANDOM_SEED
)
print(visualization_report)
print(
    (PROJECT_DIRECTORY / 'visualizations' / 'feature_interpretations.txt')
    .read_text(encoding='utf-8')
)

for filename in ('class_samples.png', 'image_feature_comparisons.png'):
    display(Image.open(PROJECT_DIRECTORY / 'visualizations' / filename))

### Dataset limitations to keep in mind

PlantVillage images are collected under controlled conditions and commonly have plain, uniform backgrounds. Field images can differ in lighting, camera quality, leaf position, and background, so this test score may not predict real-world performance. The split is stratified at the image level, not grouped by leaf or capture session. If near-duplicate images or multiple views of one leaf occur across splits, test performance could be optimistic; this notebook does not claim that such leakage has been measured. A stronger follow-up would deduplicate/group images before splitting and evaluate on an independently collected field dataset.

The three plotted descriptors (mean brightness, green-channel ratio, and edge contrast) are descriptive measurements, not causal explanations of disease or of the neural network's decisions.

## 3. Train with augmentation and compare transfer-learning stages

The training-only augmentation applies random horizontal flips, small rotations, zoom, and contrast changes. The validation and test datasets are never augmented. First, train the classifier head while the ImageNet-pretrained MobileNetV2 feature extractor is frozen. Then fine-tune the last eligible layers at a lower learning rate. Dropout, Adam, early stopping, and restored best weights provide regularization/optimization.

The comparison reports test metrics for the frozen-feature model and the validation-selected final model. Do not select settings based on test results. Training is intentionally kept small (batch size 16, five head epochs, up to three fine-tuning epochs) to be more practical on a Colab GPU.

In [ ]:
augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.10)
    ],
    name='training_only_augmentation'
)

def augment_training_batch(image_batch, label_batch):
    return augmentation(image_batch, training=True), label_batch

augmented_train = (
    datasets['train']
    .map(augment_training_batch, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

class_names = dataset_metadata['class_names']
model, base_model = build_model(
    number_of_classes=len(class_names),
    dropout_rate=0.30,
    learning_rate=1e-3
)
head_history = model.fit(
    augmented_train,
    validation_data=datasets['validation'],
    epochs=5,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=2, restore_best_weights=True
        )
    ],
    verbose=1
)
frozen_weights = model.get_weights()
frozen_validation_loss = min(head_history.history['val_loss'])

# Capture the frozen-feature test result for a like-for-like report only.
def summarize_test_metrics(current_model):
    truth = tf.concat([labels for _, labels in datasets['test']], axis=0).numpy()
    probabilities = current_model.predict(datasets['test'], verbose=0)
    predicted = probabilities.argmax(axis=1)
    loss = current_model.evaluate(datasets['test'], verbose=0, return_dict=True)['loss']
    from sklearn.metrics import accuracy_score, f1_score
    return {
        'test_loss': float(loss),
        'accuracy': float(accuracy_score(truth, predicted)),
        'macro_f1': float(f1_score(truth, predicted, average='macro', zero_division=0))
    }

frozen_test_metrics = summarize_test_metrics(model)
print('Frozen-feature test metrics:', frozen_test_metrics)

In [ ]:
unfrozen_count = unfreeze_last_layers(base_model, number_of_layers=20)
compile_model(model, learning_rate=1e-5)
fine_tune_history = model.fit(
    augmented_train,
    validation_data=datasets['validation'],
    epochs=3,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=2, restore_best_weights=True
        )
    ],
    verbose=1
)
fine_tuned_validation_loss = min(fine_tune_history.history['val_loss'])

if fine_tuned_validation_loss < frozen_validation_loss:
    selected_stage = 'fine-tuned'
else:
    model.set_weights(frozen_weights)
    selected_stage = 'frozen-only (fine-tuning did not improve validation loss)'

final_test_metrics = summarize_test_metrics(model)
comparison = pd.DataFrame(
    [
        {'stage': 'Frozen feature extractor', **frozen_test_metrics},
        {'stage': selected_stage, **final_test_metrics}
    ]
).set_index('stage')
display(comparison)
print('Selected using validation loss:', selected_stage)
print('Fine-tuned layers available:', unfrozen_count)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for history, label in ((head_history, 'Frozen head'), (fine_tune_history, 'Fine-tuning')):
    axes[0].plot(history.history['loss'], marker='o', label=f'{label} train')
    axes[0].plot(history.history['val_loss'], marker='o', linestyle='--', label=f'{label} validation')
    axes[1].plot(history.history['accuracy'], marker='o', label=f'{label} train')
    axes[1].plot(history.history['val_accuracy'], marker='o', linestyle='--', label=f'{label} validation')
axes[0].set_title('Training and validation loss')
axes[1].set_title('Training and validation accuracy')
for axis in axes:
    axis.set_xlabel('Epoch within phase')
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Final held-out evaluation and error analysis

This final evaluation reports loss, accuracy, macro/weighted precision, recall and F1, per-class results, and a confusion matrix. The following weak-class figures are from the repository's previously saved 200-images-per-class run, not a promise that this fresh Colab run will produce identical results: Tomato Early blight recall was about **0.47** and Corn Northern Leaf Blight recall was **0.70**. In that run, Tomato Early blight was most often confused with Tomato Late blight (5 of 30 test examples). These errors are consistent with visually similar symptoms as one possible explanation, but the confusion matrix alone cannot establish the cause. The next cells recompute the findings from this run.

In [ ]:
model_path = PROJECT_DIRECTORY / 'models' / 'plant_disease_classifier.keras'
model_path.parent.mkdir(parents=True, exist_ok=True)
model.save(model_path)

final_report = evaluate_model(
    model,
    datasets['test'],
    class_names,
    PROJECT_DIRECTORY
)
final_report['training_comparison'] = {
    'frozen_feature_extractor': frozen_test_metrics,
    'selected_final_model': final_test_metrics,
    'selected_stage': selected_stage,
    'frozen_validation_loss': float(frozen_validation_loss),
    'fine_tuned_validation_loss': float(fine_tuned_validation_loss),
    'fine_tuned_layers': int(unfrozen_count),
    'training_augmentation': [
        'horizontal flip', 'rotation', 'zoom', 'contrast'
    ]
}
final_report['limitations'] = [
    'PlantVillage images are collected under controlled conditions.',
    'The stratified split is image-level rather than grouped by leaf/session.',
    'Potential near-duplicate leakage was not measured in this notebook.'
]
metrics_path = PROJECT_DIRECTORY / 'models' / 'evaluation_metrics.json'
with metrics_path.open('w', encoding='utf-8') as file:
    json.dump(final_report, file, indent=2)

summary_metrics = {
    key: final_report[key]
    for key in (
        'test_loss', 'accuracy', 'macro_precision', 'macro_recall',
        'macro_f1', 'weighted_precision', 'weighted_recall', 'weighted_f1'
    )
}
display(pd.Series(summary_metrics, name='test value').to_frame())
print('Saved model:', model_path)
print('Saved metrics:', metrics_path)

In [ ]:
per_class = pd.DataFrame(final_report['classification_report']).T
per_class = per_class.loc[per_class.index.isin(class_names)]
weakest_classes = per_class.sort_values('recall').head(8)
display(weakest_classes[['precision', 'recall', 'f1-score', 'support']])

confusion = final_report['confusion_matrix']
tomato_indices = [
    index for index, name in enumerate(class_names)
    if name.startswith('Tomato___')
]
tomato_errors = []
for true_index in tomato_indices:
    for predicted_index in tomato_indices:
        if true_index != predicted_index and confusion[true_index][predicted_index] > 0:
            tomato_errors.append({
                'actual class': class_names[true_index],
                'predicted as': class_names[predicted_index],
                'count': confusion[true_index][predicted_index]
            })
tomato_errors = sorted(tomato_errors, key=lambda row: row['count'], reverse=True)
display(pd.DataFrame(tomato_errors[:10]))

confusion_path = PROJECT_DIRECTORY / 'visualizations' / 'confusion_matrix.png'
display(Image.open(confusion_path))

## 5. Predict one held-out image

The prediction example uses a test image and the same MobileNetV2 image preprocessing as the API. Its ground-truth label is shown beside the top three predictions. A single correct example is illustrative, not a measure of model performance; use the complete metrics and confusion matrix above for evaluation.

In [ ]:
import csv

manifest_path = PROJECT_DIRECTORY / 'data' / 'processed' / 'split_manifest.csv'
with manifest_path.open(newline='', encoding='utf-8') as file:
    test_rows = [row for row in csv.DictReader(file) if row['split'] == 'test']
sample = test_rows[0]
sample_path = PROJECT_DIRECTORY / sample['relative_path']
display(Image.open(sample_path))

predictor = load_predictor(PROJECT_DIRECTORY)
predictions = predictor.predict(sample_path.read_bytes(), top_k=3)
print('Ground truth:', sample['class_name'])
display(pd.DataFrame(predictions))

## Keep the results

The model and evaluation report are saved in the temporary Colab project directory under `models/`; the confusion matrix and dataset visualizations are under `visualizations/`. Before the runtime is deleted, download the artifacts you need or copy them to Google Drive. For a submission-ready notebook, use **File > Save a copy in Drive** after running the notebook so the cell outputs are preserved. Do not commit the downloaded PlantVillage dataset or TFDS cache to GitHub.

In [ ]:
# Optional: mount Drive and copy the model/report there for persistence.
# Uncomment and run only if you want to save these artifacts in your own Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# output_dir = Path('/content/drive/MyDrive/plant-disease-classification-results')
# output_dir.mkdir(parents=True, exist_ok=True)
# import shutil
# shutil.copy2(model_path, output_dir / model_path.name)
# shutil.copy2(metrics_path, output_dir / metrics_path.name)
# shutil.copy2(confusion_path, output_dir / confusion_path.name)